# Coffee and cocoa presentation

[![Launch on Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/zhittsova/coffee-cocoa-data-platform/main?labpath=notebooks/coffee-cocoa-presentation.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhittsova/coffee-cocoa-data-platform/blob/main/notebooks/coffee-cocoa-presentation.ipynb)

This notebook reads a single published DuckDB snapshot. Its charts use named dbt marts; it does not implement business transformations, train models, or select forecasts.

Binder installs the locked notebook group. Colab clones this public repository and installs that same group from the first cell. Colab needs a Python 3.12 runtime; select one under Runtime > Change runtime type if the default differs. In either service, run all cells: the first cell creates a synthetic fixture and publishes a snapshot before the charts run. It does not fetch source data.

Fixture values are for software demonstration only and are not market observations. The trade fixture covers December 2021 and January 2022. Benchmark data span January 2015 through August 2026, with Robusta missing in July and August 2026. Missing values stay missing. Forecasts use current-vintage inputs and are retrospective; their intervals are diagnostics, not prospective guarantees.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This project requires Python 3.12. In Colab, select a Python 3.12 runtime under Runtime > Change runtime type.")

try:
    import google.colab  # noqa: F401
except ImportError:
    COLAB_RUNTIME = False
else:
    COLAB_RUNTIME = True


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/coffee_cocoa_platform").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from a checked-out coffee-cocoa-data-platform repository.")


def clone_project_for_colab():
    clone_parent = Path(tempfile.mkdtemp(prefix="coffee-cocoa-repo-"))
    project_root = clone_parent / "coffee-cocoa-data-platform"
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/zhittsova/coffee-cocoa-data-platform.git", str(project_root)],
        check=True,
    )
    uv = shutil.which("uv")
    uv_version = (
        subprocess.run([uv, "--version"], capture_output=True, text=True).stdout.strip()
        if uv
        else ""
    )
    if uv_version != "uv 0.12.11":
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "uv==0.12.11"],
            check=True,
        )
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uv installation completed but the executable is not on PATH.")
    exported = subprocess.run(
        [uv, "export", "--locked", "--no-hashes", "--no-default-groups", "--group", "notebooks"],
        cwd=project_root,
        capture_output=True,
        check=True,
        text=True,
    )
    subprocess.run(
        [uv, "pip", "install", "--system", "--no-cache", "-r", "-"],
        cwd=project_root,
        input=exported.stdout,
        check=True,
        text=True,
    )
    return project_root


project_root = (
    clone_project_for_colab()
    if COLAB_RUNTIME
    else find_project_root(Path.cwd().resolve())
)
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))

root_value = os.environ.get("COFFEE_COCOA_HOME")
if not root_value:
    from coffee_cocoa_platform.notebook_fixture_cli import build_notebook_fixture

    fixture_root = Path(tempfile.mkdtemp(prefix="coffee-cocoa-notebook-"))
    os.environ["COFFEE_COCOA_HOME"] = str(fixture_root)
    print(build_notebook_fixture(fixture_root))
    root_value = str(fixture_root)

data_root = Path(root_value).expanduser().resolve()
matplotlib_config = data_root / ".matplotlib"
matplotlib_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config))

from IPython.display import display

import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from coffee_cocoa_platform.snapshots import open_snapshot

reader = open_snapshot(data_root)
connection = reader.connection
snapshot_id = reader.snapshot_id
snapshot_manifest = reader.manifest

database_path = Path(connection.execute("pragma database_list").fetchone()[2])
if snapshot_id not in database_path.parts:
    reader.close()
    raise RuntimeError("The reader did not open the selected immutable snapshot.")

source_manifests = snapshot_manifest["source_versions"]["manifests"]
source_by_name = {item["dataset"]: item for item in source_manifests}
benchmark_capture = source_by_name["benchmark_prices"]["source_capture_ids"][0]
trade_captures = source_by_name["trade"]["source_capture_ids"]
trade_capture_label = ", ".join(item[:12] for item in trade_captures)

def query_mart(sql):
    result = connection.execute(sql).to_arrow_table()
    display(result)
    return result.to_pylist()

def capture_note(capture_id):
    return capture_id[:12] if capture_id else "unknown"

def figure_note(figure, source, unit, period, coverage, vintage):
    figure.suptitle(f"{source} | units: {unit}", fontsize=11, y=0.98)
    figure.text(
        0.5,
        0.91,
        f"Period: {period} | Coverage: {coverage} | Vintage: {vintage}",
        ha="center",
        va="top",
        fontsize=8,
        wrap=True,
    )
    figure.tight_layout(rect=(0.02, 0.02, 0.98, 0.84))

benchmark_period = query_mart("""
select
    benchmark_series,
    min(month_key) as first_period,
    max(month_key) as latest_requested_period,
    max(month_key) filter (where price_usd_per_kg is not null) as latest_observed_period,
    count(*) filter (where price_usd_per_kg is not null) as observed_months
from monthly_benchmark_dynamics
group by benchmark_series
order by benchmark_series
""")

print(f"Snapshot: {snapshot_id}")
print(f"Published at: {snapshot_manifest['published_at_utc']}")
print(f"Benchmark capture: {capture_note(benchmark_capture)}")
print(f"Trade captures: {trade_capture_label}")
print(f"Warehouse copy: {database_path}")

In [ ]:
benchmark_rows = query_mart("""
select
    month_key,
    benchmark_series,
    price_usd_per_kg,
    source_capture_id
from monthly_benchmark_dynamics
order by month_key, benchmark_series
""")

series_labels = {
    "cocoa": "Cocoa",
    "coffee_arabica": "Coffee Arabica",
    "coffee_robusta": "Coffee Robusta",
}
requested_end = max(row["month_key"] for row in benchmark_rows)
last_observed = {
    row["benchmark_series"]: row["latest_observed_period"]
    for row in benchmark_period
}
partial_series = [
    series_labels[row["benchmark_series"]]
    for row in benchmark_period
    if row["latest_observed_period"] < row["latest_requested_period"]
]
partial_note = (
    f"{requested_end:%Y-%m} is partial across series: {', '.join(partial_series)} "
    "ends earlier; missing values are not filled."
    if partial_series
    else "All selected series have observations through the latest requested month."
)
coverage_note = "; ".join(
    f"{series_labels[name]} through {period:%Y-%m}" for name, period in last_observed.items()
)

figure, axis = plt.subplots(figsize=(11, 4.5))
for series_id, label in series_labels.items():
    rows = [
        row
        for row in benchmark_rows
        if row["benchmark_series"] == series_id and row["price_usd_per_kg"] is not None
    ]
    axis.plot(
        [row["month_key"] for row in rows],
        [float(row["price_usd_per_kg"]) for row in rows],
        label=label,
        linewidth=1.6,
    )
axis.set_ylabel("USD/kg")
axis.set_xlabel("Reference month")
axis.xaxis.set_major_locator(mdates.YearLocator(2))
axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis.grid(axis="y", alpha=0.25)
axis.legend(ncol=3, frameon=False)
axis.set_title(partial_note, fontsize=9)
figure_note(
    figure,
    "dbt monthly_benchmark_dynamics",
    "nominal USD/kg",
    f"2015-01 to {requested_end:%Y-%m}",
    coverage_note,
    f"synthetic fixture capture {capture_note(benchmark_capture)}; snapshot {snapshot_id}",
)
plt.show()

## Trade units, balances, and partner concentration

The trade marts use WORLD rows for product value and mass totals. The source quantity is reported in 100 kg and the staging model converts it to kg. Values stay in euros: imports use CIF and exports use FOB. A balance is shown only when both flows have the same observed CN8 leaves; it is exports FOB minus imports CIF.

Product shares describe the observed selected products in that month. The fixture is only a subset of the declared 33-leaf universe, so the chart must label that coverage as partial. It does not represent a complete coffee or cocoa market mix.

Partner concentration uses observed named partners as its denominator. Special partner amounts, WORLD totals, and reconciliation residuals stay visible as separate context. The concentration chart is not a complete-universe claim.

In [ ]:
trade_rows = query_mart("""
select
    month_key,
    product_group,
    flow_name,
    world_value_eur,
    world_net_mass_kg,
    observed_unit_value_eur_per_kg,
    observed_product_value_share,
    observed_value_product_count,
    expected_product_count,
    has_complete_value_coverage,
    has_complete_product_mix_universe
from monthly_trade_product_metrics
order by month_key, product_group, flow_name
""")

trade_months = sorted({row["month_key"] for row in trade_rows})
trade_period_label = (
    f"{trade_months[0]:%Y-%m} to {trade_months[-1]:%Y-%m}"
    if trade_months
    else "no represented trade months"
)
trade_chart_rows = [
    row for row in trade_rows
    if row["world_value_eur"] is not None
    and row["observed_product_value_share"] is not None
]
trade_labels = [
    f"{row['month_key']:%Y-%m}\n{row['product_group'].replace('_', ' ')}\n{row['flow_name']}"
    for row in trade_chart_rows
]
trade_values = [
    100 * float(row["observed_product_value_share"])
    for row in trade_chart_rows
]
incomplete_mix_rows = sum(
    not row["has_complete_product_mix_universe"] for row in trade_chart_rows
)

figure, axis = plt.subplots(figsize=(10, 4.5))
axis.bar(range(len(trade_chart_rows)), trade_values, color="#7c5c3b")
axis.set_xticks(range(len(trade_chart_rows)), trade_labels, rotation=25, ha="right")
axis.set_ylabel("Observed product value share (%)")
axis.set_title(
    f"Observed product rows only; {incomplete_mix_rows}/{len(trade_chart_rows)} "
    "rows lack complete selected-product coverage",
    fontsize=9,
)
axis.grid(axis="y", alpha=0.25)
figure_note(
    figure,
    "dbt monthly_trade_product_metrics",
    "value share (%)",
    trade_period_label,
    "WORLD rows; the fixture contains a partial subset of the declared product leaves",
    f"synthetic trade captures {trade_capture_label}; snapshot {snapshot_id}",
)
plt.show()

print("Net mass is in kg; source QUANTITY_IN_100KG was multiplied by 100.")
print("Unit values use paired positive mass and value cells and are in EUR/kg.")

In [ ]:
balance_rows = query_mart("""
select
    month_key,
    product_group,
    import_value_cif_eur,
    export_value_fob_eur,
    has_compatible_flow_coverage,
    trade_balance_eur
from monthly_trade_balances
order by month_key, product_group
""")

valid_balances = [
    row for row in balance_rows
    if row["has_compatible_flow_coverage"] and row["trade_balance_eur"] is not None
]
balance_labels = [
    f"{row['month_key']:%Y-%m}\n{row['product_group'].replace('_', ' ')}"
    for row in valid_balances
]
balance_values = [float(row["trade_balance_eur"]) for row in valid_balances]

figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(range(len(valid_balances)), balance_values, color="#397c78")
axis.axhline(0, color="#333333", linewidth=0.8)
axis.set_xticks(range(len(valid_balances)), balance_labels, rotation=20, ha="right")
axis.set_ylabel("Exports FOB minus imports CIF (EUR)")
axis.set_title(
    f"{len(valid_balances)} of {len(balance_rows)} rows have matching observed leaf coverage",
    fontsize=9,
)
axis.grid(axis="y", alpha=0.25)
figure_note(
    figure,
    "dbt monthly_trade_balances",
    "EUR",
    trade_period_label,
    "only rows with identical observed import and export CN8 leaves are plotted",
    f"synthetic trade captures {trade_capture_label}; snapshot {snapshot_id}",
)
plt.show()

concentration_rows = query_mart("""
select
    month_key,
    product_group,
    flow_name,
    named_partner_count,
    named_partner_value_eur,
    largest_named_partner_share,
    named_partner_hhi,
    observed_special_value_eur,
    world_value_eur,
    reconciliation_residual_eur,
    reconciliation_status
from monthly_partner_concentration
where named_partner_count > 0
order by month_key, product_group, flow_name
""")

concentration_chart_rows = [
    row for row in concentration_rows
    if row["largest_named_partner_share"] is not None
]
concentration_labels = [
    f"{row['month_key']:%Y-%m}\n{row['product_group'].replace('_', ' ')}\n{row['flow_name']}"
    for row in concentration_chart_rows
]
largest_shares = [
    100 * float(row["largest_named_partner_share"])
    for row in concentration_chart_rows
]

figure, axis = plt.subplots(figsize=(10, 4.5))
axis.bar(
    range(len(largest_shares)),
    largest_shares,
    color="#5079a3",
)
axis.set_xticks(range(len(concentration_labels)), concentration_labels, rotation=25, ha="right")
axis.set_ylabel("Largest observed named partner share (%)")
axis.set_title("Denominator: observed named partners only", fontsize=9)
axis.set_ylim(0, 105)
axis.grid(axis="y", alpha=0.25)
figure_note(
    figure,
    "dbt monthly_partner_concentration",
    "share of named-partner value (%)",
    trade_period_label,
    "special amounts, WORLD values, and reconciliation residuals are separate columns",
    f"synthetic trade captures {trade_capture_label}; snapshot {snapshot_id}",
)
plt.show()

## Forecast evaluation

The result mart compares persistence and seasonal naive with the AR(1) candidate. It reports the fixed development and holdout splits for one- and three-month horizons. Model selection uses development MAE, but this notebook does not repeat that logic.

The source vintage is current-vintage and the test is retrospective. Holdout errors do not prove future performance. The interval chart reports empirical coverage and the number of scored intervals; the 90% line is the nominal target, not a guarantee.

In [ ]:
forecast_rows = query_mart("""
select
    split_name,
    horizon_months,
    model_name,
    is_selected,
    origin_count,
    prediction_count,
    scored_count,
    mae,
    rmse,
    smape_percent,
    interval_scored_count,
    interval_coverage,
    mean_interval_width
from forecast_metrics
order by split_name, horizon_months, model_name
""")

forecast_identity = query_mart("""
select distinct run_id, evaluation_mode
from forecast_predictions
order by run_id, evaluation_mode
""")
forecast_run_id = forecast_identity[0]["run_id"]
evaluation_mode = forecast_identity[0]["evaluation_mode"]
holdout_rows = [row for row in forecast_rows if row["split_name"] == "holdout"]
holdout_months = query_mart("""
select
    min(origin_month) as first_origin,
    max(origin_month) as last_origin,
    min(target_month) as first_target,
    max(target_month) as last_target
from forecast_predictions
where split_name = 'holdout'
""")[0]
holdout_period = (
    f"origins {holdout_months['first_origin']:%Y-%m} to "
    f"{holdout_months['last_origin']:%Y-%m}; targets through "
    f"{holdout_months['last_target']:%Y-%m}"
)
forecast_vintage = (
    f"{evaluation_mode}; run {forecast_run_id}; "
    f"result snapshot {snapshot_id}"
)

model_order = ["persistence", "seasonal_naive", "ar1"]
model_labels = {
    "persistence": "Persistence",
    "seasonal_naive": "Seasonal naive",
    "ar1": "AR(1) candidate",
}
figure, axes = plt.subplots(2, 2, figsize=(10, 7))
for column, horizon in enumerate((1, 3)):
    horizon_rows = [
        row for row in holdout_rows if row["horizon_months"] == horizon
    ]
    for row_index, metric in enumerate(("mae", "rmse")):
        axis = axes[row_index, column]
        metric_rows = {
            row["model_name"]: row
            for row in horizon_rows
            if row[metric] is not None
        }
        names = [name for name in model_order if name in metric_rows]
        values = [float(metric_rows[name][metric]) for name in names]
        axis.bar(
            [model_labels[name] for name in names],
            values,
            color=["#557a95", "#68a691", "#bb7551"][: len(names)],
        )
        axis.set_title(f"{metric.upper()} | {horizon}-month horizon")
        axis.set_ylabel("USD/kg")
        axis.tick_params(axis="x", labelrotation=20)
        axis.grid(axis="y", alpha=0.25)
figure_note(
    figure,
    "dbt forecast_metrics",
    "USD/kg",
    holdout_period,
    "scored_count labels are reported in the table for each model and horizon",
    f"synthetic forecast fixture; {forecast_vintage}",
)
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, horizon in zip(axes, (1, 3)):
    horizon_rows = {
        row["model_name"]: row
        for row in holdout_rows
        if row["horizon_months"] == horizon
    }
    names = [name for name in model_order if name in horizon_rows]
    values = [
        100 * float(horizon_rows[name]["interval_coverage"])
        if horizon_rows[name]["interval_coverage"] is not None
        else 0
        for name in names
    ]
    axis.bar(
        [model_labels[name] for name in names],
        values,
        color=["#557a95", "#68a691", "#bb7551"][: len(names)],
    )
    axis.axhline(90, color="#a54141", linestyle="--", linewidth=1, label="90% nominal")
    axis.set_ylim(0, 100)
    axis.set_title(f"{horizon}-month horizon")
    axis.set_ylabel("Interval coverage (%)")
    axis.tick_params(axis="x", labelrotation=20)
    axis.grid(axis="y", alpha=0.25)
    axis.legend(frameon=False)
figure_note(
    figure,
    "dbt forecast_metrics",
    "interval coverage (%)",
    holdout_period,
    "interval_scored_count is the denominator; null intervals are not scored",
    f"synthetic forecast fixture; {forecast_vintage}",
)
plt.show()

reader.close()